# Phase 6: Statistical Stability Testing (5 random seeds)

Some models involve randomness during training. Random Forest picks random samples and random features for each tree, and the LSTM starts from random weights and shuffles the data. A model can therefore look a bit better or worse purely by luck. To check that our conclusions don't depend on luck, we retrain every model with **5 different random seeds** (42, 7, 123, 2024, 99) and report the **mean ± standard deviation** of MAE, RMSE and R².

We use the **tuned settings** from Phase 4, the same versions that go into the TOPSIS ranking.

**What to expect:**
- **Linear Regression** has no randomness at all, so its standard deviation is exactly 0.
- **Decision Tree**: the seed only breaks ties between equally good splits, so expect tiny or zero differences.
- **XGBoost** doesn't sub-sample rows or features at our settings, so it gives the same result for every seed (standard deviation 0).
- **LightGBM** varies a little, even though it doesn't sub-sample rows or features either. When there are more than 200,000 training rows (we have about 321,000), it decides its histogram bin edges from a random sample of 200,000 rows, and that sample depends on the seed.
- **Random Forest and LSTM** have the most built-in randomness (random samples per tree, random starting weights and shuffling).

We report whatever comes out. Small standard deviations are good news: the model gives nearly the same answer every time it's retrained.

*Why the seed-42 row here can differ slightly from Phase 4:* Phase 4 retrains its winning setting on the training rows **sorted by time** (needed for its cross-validation), while here the rows are in their normal VM-by-VM order. XGBoost and LightGBM are very slightly sensitive to row order, so the two can differ in the third decimal place. Both are legitimate; the paper reports Phase 4 for the tuned accuracy and Phase 6 for the spread.

Only the scores are kept, not the 30 trained models (5 seeds × 6 models), to save Drive space.

**Run time on free Colab:** about 30–40 minutes, mostly 5 Random Forests and 5 LSTMs. **If Colab disconnects, run everything again.** Each (model, seed) result is saved as soon as it finishes, and finished ones are skipped.

## 0. Setup

In [1]:
# Mount Google Drive (Colab) or use the local Drive folder (PC), then load our shared code.
import os, sys, importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Green_ML_Project'
except ImportError:
    PROJECT_ROOT = r'H:\My Drive\Green_ML_Project'   # running on the PC instead of Colab

os.environ['GREEN_ML_ROOT'] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))

import config as C
import utils as U
importlib.reload(C); importlib.reload(U)   # picks up any edits to the scripts without restarting

print('Project folder:', C.PROJECT_ROOT)
print('Folder found:  ', os.path.isdir(C.PROJECT_ROOT))

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Green_ML_Project
Folder found:   True


In [2]:
import gc, json, os
import numpy as np
import pandas as pd
import model_definitions as M
importlib.reload(M)

FORCE_RERUN = False
log = U.get_logger('phase06_stability_testing')
log.info(f'Seeds: {C.STABILITY_SEEDS} | CPU cores: {os.cpu_count()} | FORCE_RERUN = {FORCE_RERUN}')

RUN STARTED 2026-10-04 06:51:11 | phase06_stability_testing
python 3.13.15 | pandas 2.2.3 | numpy 2.1.3 | machine: Linux-6.6.122+-x86_64-with-glibc2.39
project root: /content/drive/MyDrive/Green_ML_Project
Seeds: [42, 7, 123, 2024, 99] | CPU cores: 2 | FORCE_RERUN = False


## 1. Load data and the tuned settings

In [3]:
train = pd.read_parquet(C.DATA_PROCESSED / 'train.parquet')
test = pd.read_parquet(C.DATA_PROCESSED / 'test.parquet')
features_main = pd.read_parquet(C.DATA_PROCESSED / 'features_main.parquet')
X_train, y_train = train[U.FEATURE_COLUMNS], train[C.TARGET].to_numpy()
X_test, y_test = test[U.FEATURE_COLUMNS], test[C.TARGET].to_numpy()

phase4 = json.load(open(C.MODELS_DIR / 'phase4_results.json'))
tuned_params = {n: (phase4[n]['best_params'] if n in phase4 else {}) for n in M.TABULAR_MODELS + ['LSTM']}
for n, p in tuned_params.items():
    log.info(f'{n:<17}: {p}')

RUNS_FILE = C.MODELS_DIR / 'phase6_runs.json'
runs = {} if FORCE_RERUN or not RUNS_FILE.exists() else json.load(open(RUNS_FILE))
if runs:
    log.info(f'{len(runs)} (model, seed) runs already finished; they will be skipped')

Linear Regression: {}
Decision Tree    : {'max_depth': 5, 'min_samples_split': 10}
Random Forest    : {'max_depth': 10, 'n_estimators': 100}
XGBoost          : {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 300}
LightGBM         : {'learning_rate': 0.05, 'max_depth': -1, 'n_estimators': 300}
LSTM             : {'hidden_size': 32, 'seq_len': 12}


## 2. Train every model with every seed

The loop goes seed by seed, so if time runs out we already have complete rounds rather than all five seeds for some models and none for others.

In [4]:
lstm_cache = {}
for seed in C.STABILITY_SEEDS:
    for name in M.TABULAR_MODELS + ['LSTM']:
        key = f'{name} | {seed}'
        if key in runs:
            continue
        params = tuned_params[name]
        with U.Timer() as t:
            if name == 'LSTM':
                seq_len = params.get('seq_len', M.LSTM_DEFAULTS['seq_len'])
                if seq_len not in lstm_cache:
                    lstm_cache[seq_len] = M.lstm_datasets(features_main, seq_len)
                ds = lstm_cache[seq_len]
                (Xtr, ytr), (Xva, yva), (Xte, yte, _) = ds['train'], ds['val'], ds['test']
                model, history = M.train_lstm(Xtr, ytr, Xva, yva, seed=seed, log_fn=lambda *_: None, **params)
                pred = M.predict_lstm(model, Xte)
                extra = {'epochs_run': len(history)}
            else:
                model = M.make_model(name, seed=seed, **params)
                model.fit(X_train, y_train)
                pred = model.predict(X_test)
                extra = {}
        scores = U.regression_metrics(y_test, pred)
        runs[key] = {'model': name, 'seed': seed, **scores, 'train_time_sec': t.seconds, **extra}
        json.dump(runs, open(RUNS_FILE, 'w'), indent=2)
        log.info(f'seed {seed:>4} | {name:<17} | MAE {scores["MAE"]:.4f} | RMSE {scores["RMSE"]:.4f} | '
                 f'R2 {scores["R2"]:.4f} | {t.seconds:.1f}s')
        del model; gc.collect()

seed   42 | Linear Regression | MAE 0.9032 | RMSE 4.5899 | R2 0.9062 | 1.7s
seed   42 | Decision Tree     | MAE 1.0502 | RMSE 4.3561 | R2 0.9156 | 2.2s
seed   42 | Random Forest     | MAE 0.6925 | RMSE 3.7283 | R2 0.9381 | 210.2s
seed   42 | XGBoost           | MAE 0.6985 | RMSE 3.7456 | R2 0.9376 | 5.6s
seed   42 | LightGBM          | MAE 0.6568 | RMSE 3.6543 | R2 0.9406 | 13.4s
seed   42 | LSTM              | MAE 0.9238 | RMSE 4.4939 | R2 0.9101 | 185.7s
seed    7 | Linear Regression | MAE 0.9032 | RMSE 4.5899 | R2 0.9062 | 0.3s
seed    7 | Decision Tree     | MAE 1.0502 | RMSE 4.3561 | R2 0.9156 | 2.5s
seed    7 | Random Forest     | MAE 0.6965 | RMSE 3.7280 | R2 0.9382 | 209.3s
seed    7 | XGBoost           | MAE 0.6985 | RMSE 3.7456 | R2 0.9376 | 5.5s
seed    7 | LightGBM          | MAE 0.6592 | RMSE 3.6342 | R2 0.9412 | 9.2s
seed    7 | LSTM              | MAE 1.0440 | RMSE 4.5021 | R2 0.9098 | 131.7s
seed  123 | Linear Regression | MAE 0.9032 | RMSE 4.5899 | R2 0.9062 | 0.3s
see

## 3. Mean ± standard deviation

The **CV (%)** columns are the *coefficient of variation*, the standard deviation as a percentage of the mean. It shows at a glance how much a model's score moves between seeds.

We also check the question that actually matters for the paper: **is the gap between the two best models bigger than the seed-to-seed noise?** If it isn't, we can't honestly say one is better than the other.

In [5]:
all_runs = pd.DataFrame(runs.values())
all_runs.to_csv(C.TABLES_DIR / 'phase6_stability_all_runs.csv', index=False)
assert all_runs.groupby('model').size().eq(len(C.STABILITY_SEEDS)).all(), 'some (model, seed) runs are missing - rerun'

order = M.TABULAR_MODELS + ['LSTM']
g = all_runs.groupby('model').agg(MAE_mean=('MAE', 'mean'), MAE_std=('MAE', 'std'),
                                  RMSE_mean=('RMSE', 'mean'), RMSE_std=('RMSE', 'std'),
                                  R2_mean=('R2', 'mean'), R2_std=('R2', 'std'),
                                  train_time_mean=('train_time_sec', 'mean')).loc[order]
g['MAE_cv_pct'] = 100 * g['MAE_std'] / g['MAE_mean']
g['RMSE_cv_pct'] = 100 * g['RMSE_std'] / g['RMSE_mean']
g.reset_index().to_csv(C.TABLES_DIR / 'phase6_stability_summary_raw.csv', index=False)
U.log_df(log, g.round(6), 'STABILITY SUMMARY (5 seeds, tuned models, test set)')

pretty = pd.DataFrame({
    'Model': g.index,
    'MAE (mean ± std)': [f'{m:.4f} ± {s:.4f}' for m, s in zip(g.MAE_mean, g.MAE_std)],
    'RMSE (mean ± std)': [f'{m:.4f} ± {s:.4f}' for m, s in zip(g.RMSE_mean, g.RMSE_std)],
    'R2 (mean ± std)': [f'{m:.4f} ± {s:.4f}' for m, s in zip(g.R2_mean, g.R2_std)],
    'MAE CV (%)': [f'{v:.2f}' for v in g.MAE_cv_pct],
    'Mean train time (s)': [f'{v:.2f}' for v in g.train_time_mean],
})
U.save_table(pretty, 'phase6_stability_summary', 'Stability across 5 random seeds (mean ± std)')
pretty

--- STABILITY SUMMARY (5 seeds, tuned models, test set) ---
                   MAE_mean   MAE_std  RMSE_mean  RMSE_std   R2_mean    R2_std  train_time_mean  MAE_cv_pct  RMSE_cv_pct
model                                                                                                                   
Linear Regression  0.903215  0.000000   4.589898  0.000000  0.906246  0.000000         0.565202    0.000000     0.000000
Decision Tree      1.050245  0.000000   4.356144  0.000000  0.915553  0.000000         2.078687    0.000000     0.000000
Random Forest      0.696531  0.003072   3.729142  0.005221  0.938113  0.000173       204.701118    0.440978     0.140002
XGBoost            0.698472  0.000000   3.745565  0.000000  0.937567  0.000000         5.767296    0.000000     0.000000
LightGBM           0.661013  0.002989   3.643508  0.007968  0.940922  0.000258         8.990417    0.452241     0.218698
LSTM               0.941834  0.060823   4.531346  0.052211  0.908613  0.002114       148.9878

,Model,MAE (mean ± std),RMSE (mean ± std),R2 (mean ± std),MAE CV (%),Mean train time (s)
0,Linear Regression,0.9032 ± 0.0000,4.5899 ± 0.0000,0.9062 ± 0.0000,0.00,0.57
1,Decision Tree,1.0502 ± 0.0000,4.3561 ± 0.0000,0.9156 ± 0.0000,0.00,2.08
2,Random Forest,0.6965 ± 0.0031,3.7291 ± 0.0052,0.9381 ± 0.0002,0.44,204.70
3,XGBoost,0.6985 ± 0.0000,3.7456 ± 0.0000,0.9376 ± 0.0000,0.00,5.77
4,LightGBM,0.6610 ± 0.0030,3.6435 ± 0.0080,0.9409 ± 0.0003,0.45,8.99
5,LSTM,0.9418 ± 0.0608,4.5313 ± 0.0522,0.9086 ± 0.0021,6.46,148.99


In [6]:
# Is the gap between the two best models (by mean MAE) larger than the seed noise?
ranked = g.sort_values('MAE_mean')
a, b = ranked.index[0], ranked.index[1]
gap = ranked.loc[b, 'MAE_mean'] - ranked.loc[a, 'MAE_mean']
noise = max(ranked.loc[a, 'MAE_std'], ranked.loc[b, 'MAE_std'])
log.info(f'Best two by mean MAE: {a} ({ranked.loc[a, "MAE_mean"]:.4f}) and {b} ({ranked.loc[b, "MAE_mean"]:.4f})')
log.info(f'Gap = {gap:.4f}; largest seed std of the two = {noise:.4f} -> gap is '
         f'{"larger" if gap > 2 * noise else "NOT clearly larger"} than 2x the seed noise')

# Does the MAE ranking of models change from seed to seed?
rank_per_seed = all_runs.pivot(index='seed', columns='model', values='MAE')[order].rank(axis=1).astype(int)
U.log_df(log, rank_per_seed, 'MAE RANK OF EACH MODEL PER SEED (1 = best)')
log.info(f'Ranking identical across all seeds: {bool((rank_per_seed.nunique() == 1).all())}')
log.info('PHASE 6 COMPLETE')
rank_per_seed

Best two by mean MAE: LightGBM (0.6610) and Random Forest (0.6965)
Gap = 0.0355; largest seed std of the two = 0.0031 -> gap is larger than 2x the seed noise
--- MAE RANK OF EACH MODEL PER SEED (1 = best) ---
model  Linear Regression  Decision Tree  Random Forest  XGBoost  LightGBM  LSTM
seed                                                                           
7                      4              6              2        3         1     5
42                     4              6              2        3         1     5
99                     4              6              3        2         1     5
123                    5              6              3        2         1     4
2024                   4              6              2        3         1     5
Ranking identical across all seeds: False
PHASE 6 COMPLETE


model,Linear Regression,Decision Tree,Random Forest,XGBoost,LightGBM,LSTM
seed,,,,,,
7,4,6,2,3,1,5
42,4,6,2,3,1,5
99,4,6,3,2,1,5
123,5,6,3,2,1,4
2024,4,6,2,3,1,5


## Done: what to check

- The **mean ± std** table goes into the paper. Zeros for Linear Regression, XGBoost and LightGBM are expected, as explained at the top.
- **"gap is larger than 2× the seed noise"**: if it says *NOT clearly larger*, the paper should call the top two models statistically tied rather than naming a single winner on accuracy.
- **Ranking per seed**: if the ranking never changes, the accuracy ordering is robust to randomness.

Next: `07_topsis_ranking.ipynb`.